In [50]:
import os, json, sys, cv2, numpy as np, torch
from pathlib import Path
VIDEO_DIR  = "Bronchoscopy_test"
ANN_DIR    = "."
OUTPUT_DIR = "./results"
CKPT_PATH  = "./checkpoints/reference_model/model-000600000.pth"
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
video_files = sorted(Path(VIDEO_DIR).rglob("input_video.mp4"))
print(f"Found {len(video_files)}  videos:")
for v in video_files:
    cap = cv2.VideoCapture(str(v))
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f"  {v.relative_to(VIDEO_DIR)}  →  {n}  frames  {n/fps:.1f}s")
sys.path.insert(0, "./alltracker")
from nets.alltracker import Net as AllTracker
model = AllTracker(seqlen=16).to(device)
ckpt  = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print("Model loaded")


Found 14  videos:
  real_seq_001_part_0_dif_1\Videos_000\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_001\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_002\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_003\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_004\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_005\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_006\video\input_video.mp4  →  24  frames  2.4s
  real_seq_001_part_0_dif_1\Videos_007\video\input_video.mp4  →  24  frames  2.4s
  stable_seq_001_part_1_dif_0\Videos_000\video\input_video.mp4  →  24  frames  2.4s
  stable_seq_001_part_1_dif_0\Videos_001\video\input_video.mp4  →  24  frames  2.4s
  stable_seq_001_part_1_dif_0\Videos_002\video\input_video.mp4  →  24  frames  2.4s
  stable_seq_001_part_1_dif_0\Videos_003\video\input_video.mp4  →  24  fra

In [51]:
import cv2
import os
import numpy as np
def save_tracking_video(video_tensor, trajs, confs, output_path, fps=10, conf_thr=0.1):
    B, T, C, H, W = video_tensor.shape
    N = trajs.shape[2]
    colors = [
        (0, 0, 255),
        (0, 255, 0),
        (255, 0, 0),
        (0, 255, 255),
        (255, 0, 255),
    ]
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))
    trajs_np = trajs[0].numpy()
    confs_np = confs[0].numpy()
    for t in range(T):
        frame = video_tensor[0, t].permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        for i in range(N):
            color = colors[i % len(colors)]
            conf  = confs_np[t, i]
            px    = int(round(trajs_np[t, i, 0]))
            py    = int(round(trajs_np[t, i, 1]))
            if not (0 <= px < W and 0 <= py < H):
                continue
            if conf > conf_thr:
                cv2.circle(frame, (px, py), 5, color, -1)
                trail_len = 20
                for t_prev in range(max(0, t - trail_len), t):
                    if confs_np[t_prev, i] <= conf_thr:
                        continue
                    px_prev = int(round(trajs_np[t_prev, i, 0]))
                    py_prev = int(round(trajs_np[t_prev, i, 1]))
                    if not (0 <= px_prev < W and 0 <= py_prev < H):
                        continue
                    alpha = (t_prev - (t - trail_len)) / trail_len
                    faded = tuple(int(c * alpha) for c in color)
                    cv2.line(frame, (px_prev, py_prev), (px, py), faded, 1)
                cv2.putText(frame, f"P{i+1}", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            else:
                cv2.circle(frame, (px, py), 5, color, 1)
                cv2.putText(frame, f"P{i+1}?", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (128,128,128), 1)
        cv2.putText(frame, f"Frame {t+1}/{T}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        writer.write(frame)
    writer.release()
    print(f"✅ videosaved → {output_path}")


In [ ]:
import os, json, cv2, pickle
import numpy as np
import torch
from pathlib import Path
import sys
sys.path.insert(0, "./alltracker")
import utils.misc
OUTPUT_DIR = "./results_bronchoscopy"
VIDEO_DIR  = "Bronchoscopy_test"
VIS_THR    = 0.6
ITERS      = 4
os.makedirs(OUTPUT_DIR, exist_ok=True)
pkl_files = sorted(Path(VIDEO_DIR).rglob("pt_gt.pkl"))
print(f"Found {len(pkl_files)}  pt_gt.pkl")
def save_tracking_video(frames_rgb, pred_trajs, gt_last, vis_last,
                        output_path, fps=10):
    T, N, _ = pred_trajs.shape
    H, W    = frames_rgb[0].shape[:2]
    colors_pred = [(0,0,255),(0,255,0),(255,0,0),(0,255,255),(255,0,255),(255,165,0),(0,165,255)]
    colors_gt   = [(0,0,180),(0,180,0),(180,0,0),(0,180,180),(180,0,180),(180,120,0),(0,120,180)]
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))
    trail_len = 15
    for t in range(T):
        frame = cv2.cvtColor(frames_rgb[t], cv2.COLOR_RGB2BGR).copy()
        for i in range(N):
            pc = colors_pred[i % len(colors_pred)]
            px = int(round(pred_trajs[t,i,0]))
            py = int(round(pred_trajs[t,i,1]))
            for t2 in range(max(0, t-trail_len), t):
                px2 = int(round(pred_trajs[t2,i,0]))
                py2 = int(round(pred_trajs[t2,i,1]))
                if 0<=px2<W and 0<=py2<H:
                    alpha = (t2-(t-trail_len)) / trail_len
                    cv2.line(frame, (px2,py2), (px,py),
                             tuple(int(c*alpha) for c in pc), 1)
            if 0<=px<W and 0<=py<H:
                cv2.circle(frame, (px,py), 5, pc, -1)
                cv2.putText(frame, f"P{i+1}", (px+6,py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.35, pc, 1)
        cv2.putText(frame, f"Frame {t+1}/{T}", (10,25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        writer.write(frame)
    writer.release()
    print(f"  Track video → {output_path}")
def save_lastframe_comparison(frame_rgb, pred_last, gt_last, vis_last,
                               metrics, output_path):
    H, W = frame_rgb.shape[:2]
    N    = len(pred_last)
    frame = cv2.cvtColor(frame_rgb.copy(), cv2.COLOR_RGB2BGR)
    colors_pred = [(0,0,255),(0,255,0),(255,0,0),(0,255,255),(255,0,255),(255,165,0),(0,165,255)]
    colors_gt   = [(0,0,180),(0,180,0),(180,0,0),(0,180,180),(180,0,180),(180,120,0),(0,120,180)]
    for i in range(N):
        pc = colors_pred[i % len(colors_pred)]
        gc = colors_gt[i % len(colors_gt)]
        px, py = int(round(pred_last[i,0])), int(round(pred_last[i,1]))
        gx, gy = int(round(gt_last[i,0])),   int(round(gt_last[i,1]))
        if 0<=px<W and 0<=py<H:
            cv2.circle(frame, (px,py), 7, pc, -1)
            cv2.putText(frame, f"P{i+1}", (px+8,py-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, pc, 1)
        if vis_last[i] and 0<=gx<W and 0<=gy<H:
            cv2.circle(frame, (gx,gy), 10, gc, 2)
            cv2.putText(frame, f"G{i+1}", (gx+8,gy+14),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, gc, 1)
            if 0<=px<W and 0<=py<H:
                cv2.line(frame, (px,py), (gx,gy), (200,200,200), 1)
    aj  = metrics.get('average_jaccard', np.nan)
    oa  = metrics.get('occlusion_accuracy', np.nan)
    da  = metrics.get('average_pts_within_thresh', np.nan)
    if hasattr(aj, '__len__'): aj = aj[0]
    if hasattr(oa, '__len__'): oa = oa[0]
    if hasattr(da, '__len__'): da = da[0]
    cv2.putText(frame,
                f"AJ={aj*100:.1f}%  OA={oa*100:.1f}%  da={da*100:.1f}%  (●Pred ○GT)",
                (10,25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
    cv2.imwrite(output_path, frame)
    print(f"  ✅ Final-frame comparison → {output_path}")
all_metrics = {}
for pkl_path in pkl_files:
    seq_name = f"{pkl_path.parts[-3]}__{pkl_path.parts[-2]}"
    print(f"\n{'='*55}")
    print(f"Processing: {seq_name}")
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    frames_np = data["video"]
    points    = data["points"].astype(np.float32)
    occluded  = data["occluded"]
    T, H, W, _ = frames_np.shape
    N = points.shape[0]
    print(f"   frames={T}, Resolution={W}x{H}, Tracked points={N}")
    pts_px = points.copy()
    pts_px[:, :, 0] *= W
    pts_px[:, :, 1] *= H
    f0_pts  = pts_px[:, 0, :]
    gt_last = pts_px[:, 1, :]
    vis_last = ~occluded[:, 1]
    video_tensor = (
        torch.tensor(frames_np)
        .permute(0,3,1,2).unsqueeze(0)
        .float().to(device)
    )
    import utils.basic
    grid_xy = utils.basic.gridcloud2d(1, H, W, norm=False, device=device).float()
    grid_xy = grid_xy.permute(0,2,1).reshape(1,1,2,H,W)
    with torch.no_grad():
        full_flows, full_visconfs, _, _ = model.forward_sliding(
            video_tensor, iters=ITERS
        )
    traj_maps = full_flows.cuda() + grid_xy
    visconf_maps = full_visconfs.cuda()
    xyt = torch.tensor(f0_pts).round().long().to(device)
    x_idx = xyt[:, 0].clamp(0, W-1)
    y_idx = xyt[:, 1].clamp(0, H-1)
    trajs_e = traj_maps[0, :, :, y_idx, x_idx].permute(2,0,1).unsqueeze(0)
    visconfs_e = visconf_maps[0, :, :, y_idx, x_idx].permute(2,0,1).unsqueeze(0)
    visconfs_e_score = visconfs_e[..., 0] * visconfs_e[..., 1]
    pred_occluded_t  = (visconfs_e_score < VIS_THR)
    query_points = np.zeros((1, N, 3), dtype=np.float32)
    query_points[0, :, 0] = 0
    query_points[0, :, 1] = f0_pts[:, 1]
    query_points[0, :, 2] = f0_pts[:, 0]
    gt_occluded_np = np.zeros((1, N, T), dtype=bool)
    gt_occluded_np[0, :, 0]   = occluded[:, 0]
    gt_occluded_np[0, :, T-1] = occluded[:, 1]
    gt_tracks_np = np.zeros((1, N, T, 2), dtype=np.float32)
    gt_tracks_np[0, :, 0,   :] = f0_pts
    gt_tracks_np[0, :, T-1, :] = gt_last
    for t in range(1, T-1):
        alpha = t / (T-1)
        gt_tracks_np[0, :, t, :] = f0_pts + alpha * (gt_last - f0_pts)
    pred_tracks_np = trajs_e.cpu().numpy()
    pred_occluded_np = pred_occluded_t.cpu().numpy()
    metrics = utils.misc.compute_tapvid_metrics(
        query_points=query_points,
        gt_occluded=gt_occluded_np,
        gt_tracks=gt_tracks_np,
        pred_occluded=pred_occluded_np,
        pred_tracks=pred_tracks_np,
        query_mode='first',
        crop_size=(H, W),
    )
    def s(v): return float(v[0]) * 100
    print(f"  OA:              {s(metrics['occlusion_accuracy']):.2f}%")
    print(f"  δ1/2/4/8/16:     "
          f"{s(metrics['pts_within_1']):.2f} / {s(metrics['pts_within_2']):.2f} / "
          f"{s(metrics['pts_within_4']):.2f} / {s(metrics['pts_within_8']):.2f} / "
          f"{s(metrics['pts_within_16']):.2f}")
    print(f"  δ_avg:           {s(metrics['average_pts_within_thresh']):.2f}%")
    print(f"  J1/2/4/8/16:     "
          f"{s(metrics['jaccard_1']):.2f} / {s(metrics['jaccard_2']):.2f} / "
          f"{s(metrics['jaccard_4']):.2f} / {s(metrics['jaccard_8']):.2f} / "
          f"{s(metrics['jaccard_16']):.2f}")
    print(f"  J_avg (AJ):      {s(metrics['average_jaccard']):.2f}%")
    all_trajs_np = pred_tracks_np[0].transpose(1,0,2)
    video_out = str(Path(OUTPUT_DIR) / f"tracking_{seq_name}.mp4")
    save_tracking_video(
        frames_rgb=list(frames_np),
        pred_trajs=all_trajs_np,
        gt_last=gt_last,
        vis_last=vis_last,
        output_path=video_out,
        fps=10,
    )
    cmp_path = str(Path(OUTPUT_DIR) / f"lastframe_{seq_name}.jpg")
    save_lastframe_comparison(
        frame_rgb=frames_np[-1],
        pred_last=all_trajs_np[-1],
        gt_last=gt_last,
        vis_last=vis_last,
        metrics=metrics,
        output_path=cmp_path,
    )
    result = {
        "seq": seq_name,
        "OA":     s(metrics['occlusion_accuracy']),
        "J_1":    s(metrics['jaccard_1']),
        "J_2":    s(metrics['jaccard_2']),
        "J_4":    s(metrics['jaccard_4']),
        "J_8":    s(metrics['jaccard_8']),
        "J_16":   s(metrics['jaccard_16']),
        "J_avg":  s(metrics['average_jaccard']),
        "d_1":    s(metrics['pts_within_1']),
        "d_2":    s(metrics['pts_within_2']),
        "d_4":    s(metrics['pts_within_4']),
        "d_8":    s(metrics['pts_within_8']),
        "d_16":   s(metrics['pts_within_16']),
        "d_avg":  s(metrics['average_pts_within_thresh']),
    }
    all_metrics[seq_name] = result
    with open(Path(OUTPUT_DIR) / f"result_{seq_name}.json", "w") as f:
        json.dump(result, f, indent=2)
keys = ["OA","J_1","J_2","J_4","J_8","J_16","J_avg",
        "d_1","d_2","d_4","d_8","d_16","d_avg"]
agg = {k: [] for k in keys}
for r in all_metrics.values():
    for k in keys:
        if not np.isnan(r[k]):
            agg[k].append(r[k])
means = {k: np.mean(v) if v else float('nan') for k, v in agg.items()}
print(f"\n{'='*110}")
print(f"{'Model':<15} {'OA':>6} | "
      f"{'J1':>6} {'J2':>6} {'J4':>6} {'J8':>6} {'J16':>6} {'Javg':>6} | "
      f"{'d1':>6} {'d2':>6} {'d4':>6} {'d8':>6} {'d16':>6} {'davg':>6} | C")
sep = "-" * 110
print(sep)
ref = {
    "CoTracker2":  [78.59, 2.20, 2.20, 5.48, 18.26,35.60,12.75, 3.46, 3.46, 9.62,26.15,45.00,17.54, "/"],
    "CoTracker3":  [69.10, 2.06, 4.62, 8.13, 22.64,44.62,16.41, 3.46, 7.31,12.31,28.08,65.51,23.33, "80.39"],
    "EndoTracker": [67.56, 5.86, 8.24,12.46, 24.78,45.77,19.42, 8.85,14.23,19.62,40.13,63.21,29.21, "84.65*"],
}
for name, v in ref.items():
    print(f"{name:<15} {v[0]:>6.2f} | "
          f"{v[1]:>6.2f} {v[2]:>6.2f} {v[3]:>6.2f} {v[4]:>6.2f} {v[5]:>6.2f} {v[6]:>6.2f} | "
          f"{v[7]:>6.2f} {v[8]:>6.2f} {v[9]:>6.2f} {v[10]:>6.2f} {v[11]:>6.2f} {v[12]:>6.2f} | {v[13]:>6}")
print(sep)
print(f"{'AllTracker':<15} {means['OA']:>6.2f} | "
      f"{means['J_1']:>6.2f} {means['J_2']:>6.2f} {means['J_4']:>6.2f} "
      f"{means['J_8']:>6.2f} {means['J_16']:>6.2f} {means['J_avg']:>6.2f} | "
      f"{means['d_1']:>6.2f} {means['d_2']:>6.2f} {means['d_4']:>6.2f} "
      f"{means['d_8']:>6.2f} {means['d_16']:>6.2f} {means['d_avg']:>6.2f} |   /")
print(sep)
summary = {"clips": all_metrics, "mean": means}
with open(Path(OUTPUT_DIR) / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nResults saved to  {OUTPUT_DIR}/")


Found 13 pt_gt.pkl files

Processing: real_seq_001_part_0_dif_1__Videos_000
   frames=24, Resolution=480x480, Tracked points=5
  OA:              85.22%
  δ1/2/4/8/16:     1.75 / 7.02 / 14.91 / 29.82 / 57.02
  δ_avg:           22.11%
  J1/2/4/8/16:     0.96 / 3.94 / 8.76 / 19.21 / 34.39
  J_avg (AJ):      13.45%
  ✅ Tracking video → results_bronchoscopy\tracking_real_seq_001_part_0_dif_1__Videos_000.mp4
  ✅ Final-frame comparison → results_bronchoscopy\lastframe_real_seq_001_part_0_dif_1__Videos_000.jpg

Processing: real_seq_001_part_0_dif_1__Videos_001
   frames=24, Resolution=480x480, Tracked points=5
  OA:              44.35%
  δ1/2/4/8/16:     0.00 / 0.00 / 1.77 / 7.96 / 23.89
  δ_avg:           6.73%
  J1/2/4/8/16:     0.00 / 0.00 / 0.00 / 2.53 / 11.72
  J_avg (AJ):      2.85%
  ✅ Tracking video → results_bronchoscopy\tracking_real_seq_001_part_0_dif_1__Videos_001.mp4
  ✅ Final-frame comparison → results_bronchoscopy\lastframe_real_seq_001_part_0_dif_1__Videos_001.jpg

Processing: